# Stage 3 — Finance DPO Preference Alignment
This standalone notebook creates an SFT warm start and then performs DPO preference alignment on Finance FAQ responses. It includes SFT-vs-DPO evaluation, adapter saving, and a Finance-only ipywidgets demo. The SFT warm start is included because DPO should begin from an instruction-tuned model.

In [ ]:
# Install pinned, Colab-compatible packages
!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib


In [ ]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


In [ ]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


In [ ]:
# Define shared Step 10 question set and base-model outputs before any stage-3 training.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=240, temperature=0.2):
    prompt = f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in base_answers.items():
    print("\nQ:", q, "\nBase:", a)

## Build SFT warm-start dataset

In [ ]:
instruction_rows=[json.loads(x) for x in (DATA_DIR/"instruction_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def sft_text(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token}"}
sft_dataset=Dataset.from_list([sft_text(r) for r in instruction_rows])
print("SFT examples:",len(sft_dataset))

In [ ]:
from trl import SFTTrainer, SFTConfig
SFT_DIR=Path("/content/finance_adapters/stage3_sft_warm_start")
sft_args=SFTConfig(output_dir=str(SFT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=10,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
sft_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=sft_dataset,args=sft_args)
sft_trainer.train(); model.save_pretrained(str(SFT_DIR)); tokenizer.save_pretrained(str(SFT_DIR))
print("Warm-start adapter saved:",SFT_DIR)

## SFT evaluation snapshot

In [ ]:
FastLanguageModel.for_inference(model)
sft_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in sft_answers.items():
    print("\nQ:", q, "\nSFT:", a)

## Build DPO dataset

In [ ]:
preference_rows=[json.loads(x) for x in (DATA_DIR/"preference_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def dpo_row(r):
    prompt=f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['prompt'].strip()}\n\n### Assistant:\n"
    return {"prompt":prompt,"chosen":r['chosen'].strip()+tokenizer.eos_token,"rejected":r['rejected'].strip()+tokenizer.eos_token}
dpo_dataset=Dataset.from_list([dpo_row(r) for r in preference_rows])
print("Preference examples:",len(dpo_dataset)); print(dpo_dataset[0])

## Train DPO alignment

In [ ]:
from trl import DPOTrainer, DPOConfig
model.train()
DPO_DIR=Path("/content/finance_adapters/stage3_dpo")
dpo_args=DPOConfig(output_dir=str(DPO_DIR),per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=5e-6,beta=0.1,max_length=MAX_SEQ_LENGTH,max_prompt_length=512,logging_steps=5,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
dpo_trainer=DPOTrainer(model=model,ref_model=None,args=dpo_args,train_dataset=dpo_dataset,processing_class=tokenizer)
dpo_trainer.train(); model.save_pretrained(str(DPO_DIR)); tokenizer.save_pretrained(str(DPO_DIR))
print("DPO adapter saved:",DPO_DIR)

## Step 10 - Final evaluation (Base vs SFT vs DPO)

In [ ]:
import pandas as pd
from IPython.display import display

FastLanguageModel.for_inference(model)
dpo_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}

df_final = pd.DataFrame([
    {
        "Question": q,
        "Base Model Answer": base_answers[q],
        "SFT Model Answer": sft_answers[q],
        "DPO Model Answer": dpo_answers[q],
        "Best Answer": "TODO",
        "Reason": "TODO",
    }
    for q in EVAL_QUESTIONS
])
display(df_final)

# Export Step 10 table to reports/final_evaluation.md
report_candidates = [Path("../reports"), Path("./reports"), Path("/content/reports")]
reports_dir = next((p for p in report_candidates if p.exists()), report_candidates[0])
reports_dir.mkdir(parents=True, exist_ok=True)
out_path = reports_dir / "final_evaluation.md"

lines = [
    "# Final Evaluation (Base vs SFT vs DPO)",
    "",
    "This report compares all three stages of model quality:",
    "",
    "1. Base model",
    "2. Instruction fine-tuned (SFT) model",
    "3. DPO-aligned model",
    "",
    "## Evaluation Criteria",
    "",
    "- Correctness",
    "- Helpfulness",
    "- Domain accuracy",
    "- Safety",
    "- Tone",
    "- Clarity",
    "- Hallucination reduction",
    "- Professional response quality",
    "",
    "## Final Comparison Table",
    "",
    "| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |",
    "| --- | --- | --- | --- | --- | --- |",
]
for q in EVAL_QUESTIONS:
    base = " ".join(base_answers[q].split()).replace("|", "\\|")
    sft = " ".join(sft_answers[q].split()).replace("|", "\\|")
    dpo = " ".join(dpo_answers[q].split()).replace("|", "\\|")
    lines.append(f"| {q} | {base} | {sft} | {dpo} | TODO | TODO |")

lines += [
    "",
    "## Overall Findings",
    "",
    "- Stage with largest quality jump: TODO",
    "- Safety and professionalism improvements from DPO: TODO",
    "- Final model limitations: TODO",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 10 report written:", out_path.resolve())

## Interactive demo

In [ ]:
# Finance-only ipywidgets demo
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)
